# E-Methanol Reactor Predictive Surrogate Model

This notebook trains a Machine Learning (Random Forest) surrogate model using the synthetic data generated from our rigorous 1D physics-based reactor model. 

It allows for **instantaneous prediction** of reactor performance (CO2 conversion, Methanol Selectivity, and Space-Time Yield) without needing to solve the complex ODEs.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Load the DOE dataset generated by our 1D model
df = pd.read_csv('../outputs/membrane_doe.csv')
print(f"Loaded {len(df)} simulated reactor cases.")
df.head()

Matplotlib is building the font cache; this may take a moment.


Loaded 200 simulated reactor cases.


,inlet_temperature_k,inlet_pressure_bar,inlet_flow_mol_s,h2_co2_ratio,length_m,water_permeance_mol_m2_s_pa,sweep_water_partial_pressure_bar,membrane_enabled,isothermal,co2_conversion,methanol_selectivity_carbon,methanol_sty_kg_m3cat_h,outlet_temperature_k,peak_temperature_k,outlet_pressure_bar,pressure_drop_bar,water_removed_fraction,h2_loss_fraction,data_source
0,506.906683,64.860690,0.025065,3.225207,0.950249,3.055209e-07,0.000010,True,False,0.328314,7.687585e-08,0.000096,490.492294,506.906683,64.852873,0.007817,0.891460,0.022768,simulated_doe
1,520.635989,59.853471,0.018295,3.303032,0.917638,2.711929e-08,0.000216,True,False,0.164715,2.011705e-07,0.000094,492.781439,520.635989,59.848064,0.005407,0.364954,0.002460,simulated_doe
2,498.468378,47.674868,0.029901,3.792662,1.433269,4.799901e-07,0.000044,True,False,0.412187,1.508791e-07,0.000166,490.836776,498.468378,47.655387,0.019481,0.930174,0.033355,simulated_doe
3,474.364842,50.626980,0.008967,3.035680,1.272333,6.202264e-08,0.005643,True,False,0.265609,1.752161e-07,0.000050,492.619108,492.619108,50.623809,0.003171,0.813344,0.013470,simulated_doe
4,507.195838,45.705882,0.018931,3.247515,0.517691,2.123653e-08,0.001192,True,False,0.142989,1.644225e-07,0.000124,492.915602,507.195838,45.701625,0.004257,0.140488,0.000802,simulated_doe


## 1. Train the Machine Learning Surrogate Models
We will train a separate Random Forest for each key performance indicator (Target).

In [ ]:
features = [
    'inlet_temperature_k',
    'inlet_pressure_bar',
    'inlet_flow_mol_s',
    'h2_co2_ratio',
    'length_m',
    'water_permeance_mol_m2_s_pa',
    'sweep_water_partial_pressure_bar'
]

targets = [
    'co2_conversion',
    'methanol_selectivity_carbon',
    'methanol_sty_kg_m3cat_h'
]

X = df[features]
y = df[targets]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {}

for target in targets:
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train[target])
    models[target] = rf
    
    # Evaluate
    preds = rf.predict(X_test)
    r2 = r2_score(y_test[target], preds)
    print(f"{target}: R² = {r2:.3f}")


## 2. Interactive Prediction Interface
Input your own reactor parameters to instantly predict the production results.

In [ ]:
def predict_reactor_performance(T_in, P_in, flow, ratio, length, permeance, sweep_P):
    """
    Predicts the reactor performance using the trained ML models.
    """
    input_data = pd.DataFrame([{
        'inlet_temperature_k': T_in,
        'inlet_pressure_bar': P_in,
        'inlet_flow_mol_s': flow,
        'h2_co2_ratio': ratio,
        'length_m': length,
        'water_permeance_mol_m2_s_pa': permeance,
        'sweep_water_partial_pressure_bar': sweep_P
    }])
    
    print("--- ML Predicted Reactor Performance ---")
    for target, model in models.items():
        pred = model.predict(input_data)[0]
        if 'sty' in target:
            print(f"Methanol Production (STY): {pred:.4f} kg/(m³·h)")
        elif 'conversion' in target:
            print(f"CO2 Conversion:            {pred*100:.2f}%")
        elif 'selectivity' in target:
            print(f"Methanol Selectivity:      {pred*100:.2f}%")

# --- TEST THE PREDICTOR HERE ---
predict_reactor_performance(
    T_in=493.15,      # 220 °C
    P_in=50.0,        # bar
    flow=0.015,       # mol/s
    ratio=3.5,        # H2:CO2 ratio
    length=1.5,       # meters
    permeance=1e-7,   # mol/(m²·s·Pa)
    sweep_P=1e-4      # bar
)

## 3. Feature Importance Analysis
Let's see which operating conditions have the biggest impact on Methanol Production.

In [ ]:
target = 'methanol_sty_kg_m3cat_h'
importances = models[target].feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 5))
plt.title("Impact of Operating Conditions on Methanol Production")
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

## 4. Predictive Model Performance (Predicted vs Actual)
Let's visualize how accurately our Machine Learning models predict the true rigorous physics simulations. Points closer to the dashed diagonal line indicate perfect predictions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, target in zip(axes, targets):
    y_pred = models[target].predict(X_test)
    ax.scatter(y_test[target], y_pred, alpha=0.6, color='seagreen', edgecolor='k')
    
    # Diagonal reference line
    min_val = min(y_test[target].min(), y_pred.min())
    max_val = max(y_test[target].max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='Perfect Prediction')
    
    ax.set_title(f'{target.replace("_", " ").title()}')
    ax.set_xlabel('Actual (1D Physics Simulation)')
    ax.set_ylabel('Predicted (Random Forest)')
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.7)

plt.tight_layout()
plt.show()
